# img2txt: дерматоскопический пайплайн

**Пайплайн из 4 шагов, каждый принимает и возвращает DataFrame:**

1. **Извлечение признаков** — сегментация + 60+ признаков (цвет, форма, граница, текстура)
2. **Бакетирование** — числовые признаки -> категориальные метки
3. **Ранжирование** — нейросеть выбирает топ-10 важных признаков
4. **Генерация текста** — Mistral-7B генерирует клиническое описание на русском

---

# Часть 1. Обучение / исследование (batch)

In [ ]:
import pandas as pd
import torch

from extraction.feature_extraction_batch import extract_features_batch, images_to_df
from analysis.feature_bucketing_batch import bucket_features_batch, get_label_statistics
from importance.importance_inference import rank_features_batch
from generation.description_inference import generate_descriptions_batch
from generation.classification_types import ClassificationResult, Structure, FeatureType

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Пути (Kaggle)
IMAGE_DIR = "/kaggle/input/datasets/mihailodin1/all-image-skin"
YOLO_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_yolo.pt"
UNET_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_unet.pth"
IMPORTANCE_CHECKPOINT = "importance_checkpoints/best.pt"
FEATURES_CSV = "/kaggle/input/datasets/mihailodin1/features-img2txt/features_dataset.csv"

## Шаг 1. Извлечение признаков

In [ ]:
# Вариант A: извлечь признаки из директории с изображениями
df = images_to_df(IMAGE_DIR)
df = extract_features_batch(df, yolo_weights=YOLO_WEIGHTS, unet_weights=UNET_WEIGHTS)
df.to_csv("features_dataset.csv", index=False)
print(f"Извлечено: {len(df)} изображений, успешно: {(df['status'] == 'success').sum()}")

In [ ]:
# Вариант B: загрузить ранее вычисленные признаки
df = pd.read_csv(FEATURES_CSV)
print(f"Загружено: {len(df)} изображений")

## Шаг 2. Бакетирование признаков

In [ ]:
df = bucket_features_batch(df)
print(f"Добавлены колонки: labels, features_organized, labels_json")

In [ ]:
# Статистика по меткам
stats = get_label_statistics(df)
if not stats.empty:
    print(stats.to_string())

## Шаг 3. Ранжирование важных признаков

### 3a. Генерация псевдо-меток (для обучения модели)

In [ ]:
from importance.pseudo_importance import add_pseudo_important_labels

df = add_pseudo_important_labels(
    df,
    labels_col="labels",
    features_col="features_json",
    top_k=15,
    mode="combined",
    z_weight=1.5,
    prior_weight=0.8,
)
df["important_labels"] = df["pseudo_important_labels"]

print("Пример псевдо-меток:")
print(df["pseudo_important_labels"].iloc[0])

### 3b. Формирование датасета для обучения (реальные + псевдо, без пересечений)

In [ ]:
from pathlib import Path

ANNOTATIONS_CSV = "annotations.csv"  # реальная разметка

# Загружаем реальные аннотации
df_annot = pd.read_csv(ANNOTATIONS_CSV)

# Нормализуем пути: сравниваем только по имени файла (без директории)
# чтобы не зависеть от того, абсолютный путь или относительный
def _fname(p):
    return Path(str(p)).name

annot_map = {_fname(row["image_path"]): row["important_labels"]
             for _, row in df_annot.iterrows()}

# Подставляем реальные метки там, где есть аннотация,
# для остальных оставляем псевдо-метки из cell-9
def _pick_labels(row):
    key = _fname(row["image_path"])
    return annot_map[key] if key in annot_map else row["pseudo_important_labels"]

df["important_labels"] = df.apply(_pick_labels, axis=1)

n_real   = sum(1 for p in df["image_path"] if _fname(p) in annot_map)
n_pseudo = len(df) - n_real
print(f"Реальных аннотаций : {n_real}")
print(f"Псевдо-меток       : {n_pseudo}")
print(f"Итого              : {len(df)}")

# Сохраняем два CSV:
# 1. Только реальная разметка — используется как data_csv в train_importance
real_mask = df["image_path"].apply(_fname).isin(annot_map)
df_real_only = df[real_mask].copy()
df_real_only.to_csv("features_dataset_real.csv", index=False)

# 2. Только псевдо-разметка (без пересечений) — используется как pseudo_csv
df_pseudo_only = df[~real_mask].copy()
df_pseudo_only.to_csv("features_dataset_pseudo.csv", index=False)

print("\nСохранено:")
print(f"  features_dataset_real.csv   — {len(df_real_only)} строк (реальные)")
print(f"  features_dataset_pseudo.csv — {len(df_pseudo_only)} строк (псевдо, без пересечений)")

### Диагностика утечки RAM (tracemalloc)

Запускает фоновый поток, который каждые 30 секунд печатает топ-15 строк, аллоцирующих больше всего памяти, и diff относительно базового снимка. Запусти эту ячейку **до** ячейки обучения.

In [ ]:
import tracemalloc, threading, time, gc, os

# Останавливаем предыдущий мониторинг, если был
_TM_STOP = globals().get("_TM_STOP")
if _TM_STOP is not None:
    _TM_STOP.set()
    time.sleep(0.1)

tracemalloc.stop()
tracemalloc.start(25)  # 25 кадров стека на аллокацию
gc.collect()
_baseline = tracemalloc.take_snapshot()
_TM_STOP = threading.Event()

def _proc_rss_mb():
    try:
        with open(f"/proc/{os.getpid()}/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024
    except Exception:
        return float("nan")

def _monitor(interval=30, top_n=15):
    step = 0
    while not _TM_STOP.wait(interval):
        step += 1
        gc.collect()
        snap = tracemalloc.take_snapshot()
        diff = snap.compare_to(_baseline, "lineno")
        rss = _proc_rss_mb()
        print(f"\n=== tracemalloc tick #{step}  RSS={rss:.0f} MB ===")
        for stat in diff[:top_n]:
            frame = stat.traceback[0]
            print(f"  +{stat.size_diff/1e6:+7.1f} MB  size={stat.size/1e6:6.1f} MB  count_diff={stat.count_diff:+d}")
            print(f"      {frame.filename}:{frame.lineno}")
        # топ-3 покажем с полным стеком — там видно, кто реально вызывает аллокацию
        print("  -- top-3 traceback:")
        for stat in diff[:3]:
            print(f"  +{stat.size_diff/1e6:+.1f} MB at:")
            for line in stat.traceback.format()[-6:]:
                print("     " + line)

_thread = threading.Thread(target=_monitor, daemon=True)
_thread.start()
print(f"tracemalloc started, baseline RSS={_proc_rss_mb():.0f} MB. Run training cell now.")
print("Чтобы остановить мониторинг: _TM_STOP.set()")


In [ ]:
from importance.train_importance import train_importance

best_score = train_importance(
    data_csv="features_dataset_real.csv",
    image_dir=IMAGE_DIR,
    pseudo_csv="features_dataset_pseudo.csv",
    pseudo_weight=0.3,
    backbone="efficientnet_b0",
    epochs=50,
    batch_size=64,          # 32/GPU × 2 GPU
    lr=1e-4,
    freeze_backbone_epochs=5,
    label_smoothing=0.05,
    num_workers=4,          # параллельная загрузка данных
    use_amp=True,           # mixed precision на T4
    multi_gpu=True,         # использовать обе GPU через DataParallel
    out_dir="importance_checkpoints",
)
print(f"Best validation score: {best_score:.4f}")

### 3c. Ранжирование обученной моделью (batch)

In [ ]:
df = rank_features_batch(df, importance_model_path=IMPORTANCE_CHECKPOINT, device=device)

print("Пример important_labels:")
print(df["important_labels"].iloc[0])

## Шаг 4. Генерация клинических описаний (Mistral-7B)

Первый вызов загружает модель (~14GB). Последующие используют кэш.

In [ ]:
df = generate_descriptions_batch(df, device=device)

print("Пример описания:")
print(df["description"].iloc[0])

In [ ]:
# Сохранение результатов
output_csv = "features_with_descriptions.csv"
df.to_csv(output_csv, index=False)
print(f"Сохранено: {len(df)} строк в {output_csv}")

---

# Часть 2. Инференс (single image)

Тот же пайплайн, но для одного изображения — однострочный DataFrame.

In [ ]:
image_path = "/path/to/lesion.jpg"

# Шаг 1: извлечение признаков
df_single = pd.DataFrame([{"image_path": image_path}])
df_single = extract_features_batch(df_single, yolo_weights=YOLO_WEIGHTS, unet_weights=UNET_WEIGHTS, verbose=False)

# Шаг 2: бакетирование
df_single = bucket_features_batch(df_single, verbose=False)

# Шаг 3: ранжирование
df_single = rank_features_batch(df_single, importance_model_path=IMPORTANCE_CHECKPOINT, device=device, verbose=False)

# Шаг 4: генерация описания (с классификацией)
df_single["classification"] = ClassificationResult(
    feature_type=FeatureType.SINGLE,
    structure=Structure.GLOBULES,
    properties=["однородный"],
    final_class="Меланома",
)
df_single = generate_descriptions_batch(df_single, classification_col="classification", device=device, verbose=False)

print(df_single.iloc[0]["description"])